In [2]:
# Celda 1 — Imports y carga de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8')

df = pd.read_csv("../data/framingham_heart_study.csv")
print(f"✅ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nValores nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

✅ Dataset cargado: 4240 filas, 16 columnas

Valores nulos:
education     105
cigsPerDay     29
BPMeds         53
totChol        50
BMI            19
heartRate       1
glucose       388
dtype: int64


In [4]:
# Celda 2 — Imputación de valores nulos
df_clean = df.copy()

cols_mediana = ['cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose']
for col in cols_mediana:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

cols_moda = ['education', 'BPMeds']
for col in cols_moda:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"✅ Nulos restantes: {df_clean.isnull().sum().sum()}")

✅ Nulos restantes: 0


In [5]:
# Celda 3 — Normalización y split train/test
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df_clean.drop('TenYearCHD', axis=1)
y = df_clean['TenYearCHD']

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalización
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Split completado")
print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"Positivos en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Positivos en test:  {y_test.sum()} ({y_test.mean()*100:.1f}%)")

✅ Split completado
Train: 3392 muestras
Test:  848 muestras
Positivos en train: 515 (15.2%)
Positivos en test:  129 (15.2%)


In [6]:
# Celda 4 — SMOTE para balancear clases
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

print(f"✅ SMOTE aplicado")
print(f"Antes — Positivos: {y_train.sum()} / Negativos: {(y_train==0).sum()}")
print(f"Después — Positivos: {y_train_sm.sum()} / Negativos: {(y_train_sm==0).sum()}")

✅ SMOTE aplicado
Antes — Positivos: 515 / Negativos: 2877
Después — Positivos: 2877 / Negativos: 2877


In [7]:
# Celda 5 — Guardar datos procesados
import numpy as np

np.save('../data/X_train.npy', X_train_sm)
np.save('../data/X_test.npy', X_test_scaled)
np.save('../data/y_train.npy', y_train_sm)
np.save('../data/y_test.npy', y_test.values)

df_clean.to_csv('../data/framingham_clean.csv', index=False)

print("✅ Datos guardados en /data:")
print("   - X_train.npy  (con SMOTE)")
print("   - X_test.npy")
print("   - y_train.npy")
print("   - y_test.npy")
print("   - framingham_clean.csv")

✅ Datos guardados en /data:
   - X_train.npy  (con SMOTE)
   - X_test.npy
   - y_train.npy
   - y_test.npy
   - framingham_clean.csv


In [8]:
# Celda 6 — Conclusiones
print("""
=== CONCLUSIONES FEATURE ENGINEERING ===

1. IMPUTACIÓN:
   - Variables continuas → mediana (robusta a outliers)
   - Variables categóricas → moda
   - Resultado: 0 nulos restantes ✅

2. SPLIT:
   - Train: 3,392 muestras (80%)
   - Test:  848 muestras (20%)
   - Estratificado → misma proporción en ambos sets ✅

3. NORMALIZACIÓN:
   - StandardScaler aplicado
   - Fit solo en train, transform en ambos ✅

4. SMOTE:
   - Antes: 515 positivos vs 2,877 negativos
   - Después: 2,877 vs 2,877 (balanceado) ✅

5. PRÓXIMO PASO: Notebook 03 — Modelos ML
""")


=== CONCLUSIONES FEATURE ENGINEERING ===

1. IMPUTACIÓN:
   - Variables continuas → mediana (robusta a outliers)
   - Variables categóricas → moda
   - Resultado: 0 nulos restantes ✅

2. SPLIT:
   - Train: 3,392 muestras (80%)
   - Test:  848 muestras (20%)
   - Estratificado → misma proporción en ambos sets ✅

3. NORMALIZACIÓN:
   - StandardScaler aplicado
   - Fit solo en train, transform en ambos ✅

4. SMOTE:
   - Antes: 515 positivos vs 2,877 negativos
   - Después: 2,877 vs 2,877 (balanceado) ✅

5. PRÓXIMO PASO: Notebook 03 — Modelos ML

